# 54 — Woman/man differential steering (greedy tier)

Evaluates both FRH gender directions (`woman − man` and `man − woman`)
over every prompt in `long-generation-inputs.yaml` at the established
24-step horizon. The positive-only control separates attraction from
repulsion; frame difference is the paper operation, and normalized
signed score voting is the score-space control.

This is a descriptive steering audit, not a normative comparison of
genders. Qualitative outputs require explicit stereotype review.
Generation is disabled by default and existing logs resume by cell key.


In [1]:
# Papermill parameters
RUN_GENERATION = False
SMOKE = False
SMOKE_STEPS = 8
OVERWRITE = False
CONFIRM_FULL_RUN = ""
RUN_TRAJECTORIES = False


In [2]:
# Parameters
RUN_GENERATION = True
SMOKE = False
OVERWRITE = False
CONFIRM_FULL_RUN = "RUN_54_60_CELLS"
RUN_TRAJECTORIES = True


In [3]:
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

from frames.evaluation.harness import EvaluationHarness
from frames.experiments.long_generation import (
    generation_metadata,
    load_long_generation_experiment,
    token_bins,
)
from frames.experiments.notebook_helpers import llama_chat
from frames.linalg.angles import vector_subspace_cosine
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept

CONFIG_PATH = Path("long-generation-inputs.yaml")
EXPERIMENT = load_long_generation_experiment(CONFIG_PATH, "experiment_54")

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"
MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
METHODS = ("attract-only", "frame-diff", "score-neg")
BIN_WIDTH = 20

model_max_memory = None
if raw_max_memory := os.getenv("FRH_MODEL_MAX_MEMORY"):
    model_max_memory = {
        int(device): limit
        for device, limit in json.loads(raw_max_memory).items()
    }
    print(f"model memory caps: {model_max_memory}")
model_device_map = os.getenv("FRH_MODEL_DEVICE_MAP", "auto")
print(f"model device map: {model_device_map}")

prompts_by_id = {
    row["id"]: llama_chat(row["user"], row["assistant_prefix"])
    for row in EXPERIMENT.prompts
}
active_prompts = list(EXPERIMENT.prompts)
active_directions = list(EXPERIMENT.negative_directions)
active_lambdas = list(EXPERIMENT.lambdas)
run_steps = EXPERIMENT.steps
if SMOKE:
    active_prompts = active_prompts[:1]
    active_directions = active_directions[:1]
    run_steps = min(SMOKE_STEPS, EXPERIMENT.steps)

full_cells = EXPERIMENT.guided_run_count(len(METHODS))
active_cells = (
    len(active_prompts)
    * len(active_directions)
    * len(METHODS)
    * len(active_lambdas)
)
confirmation = f"RUN_54_{full_cells}_CELLS"
print(
    f"full matrix: {len(EXPERIMENT.negative_directions)} directions x "
    f"{len(EXPERIMENT.prompts)} prompts x {len(METHODS)} guided methods x "
    f"{len(EXPERIMENT.lambdas)} lambdas = {full_cells} guided continuations"
)
print(
    f"nominal full token budget: "
    f"{EXPERIMENT.output_token_budget(len(METHODS)):,} (early EOS excluded)"
)
print(
    f"this execution: {active_cells} cells at steps={run_steps}; "
    f"SMOKE={SMOKE}, RUN_GENERATION={RUN_GENERATION}"
)
print(f"full-run confirmation: {confirmation}")

if RUN_GENERATION and not SMOKE and CONFIRM_FULL_RUN != confirmation:
    raise RuntimeError(
        "Full generation is locked. Set CONFIRM_FULL_RUN to the printed string."
    )
if SMOKE_STEPS < 1:
    raise ValueError("SMOKE_STEPS must be positive")


model memory caps: {0: '14.5GiB', 1: '2GiB'}
model device map: sequential
full matrix: 2 directions x 10 prompts x 3 guided methods x 1 lambdas = 60 guided continuations
nominal full token budget: 1,440 (early EOS excluded)
this execution: 60 cells at steps=24; SMOKE=False, RUN_GENERATION=True
full-run confirmation: RUN_54_60_CELLS


In [4]:
suffix = "_smoke" if SMOKE else ""
LOG_PATH = Path(f"resources/54_long_negative{suffix}.jsonl")
BASELINE_PATH = Path(f"resources/54_long_negative_baselines{suffix}.json")
SUMMARY_PATH = Path(f"resources/54_long_negative_summary{suffix}.csv")
TRAJECTORY_PATH = Path(f"resources/54_long_negative_trajectories{suffix}.csv")
TRAJECTORY_FIGURE_PATH = Path(
    f"resources/54_long_negative_trajectories{suffix}.png"
)
FIGURE_PATH = Path(f"resources/54_long_negative_quadrants{suffix}.png")

def read_records(path):
    if not path.exists() or not path.read_text().strip():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line]


def record_key(config):
    return (
        config["direction_id"],
        config["prompt_id"],
        config["method"],
        float(config["lam"]),
        int(config["steps"]),
    )


if RUN_GENERATION and OVERWRITE:
    for path in (LOG_PATH, BASELINE_PATH, SUMMARY_PATH, TRAJECTORY_PATH):
        path.unlink(missing_ok=True)

planned = {
    (direction["id"], prompt["id"], method, float(lam), run_steps)
    for direction in active_directions
    for prompt in active_prompts
    for method in METHODS
    for lam in active_lambdas
}
completed = {record_key(row["config"]) for row in read_records(LOG_PATH)}
unexpected = completed - planned
if unexpected:
    raise RuntimeError(
        f"{LOG_PATH} contains {len(unexpected)} cells outside the resolved "
        "matrix; preserve it and start a new artifact or use OVERWRITE=True"
    )
print(f"resume state: {len(completed)}/{active_cells} active cells already logged")


resume state: 0/60 active cells already logged


## Model, concepts, and shared baselines

`prompt_ids: all` is resolved in catalog order before this cell. Baselines
are generated once per prompt and shared across both steering directions.


In [5]:
need_model = RUN_GENERATION or RUN_TRAJECTORIES
fur = harness = None
concepts = {}
baseline_texts_by_id = {}

if need_model:
    fur = FrameUnembeddingRepresentation.from_model_id(
        id=MODEL_ID,
        device_map=model_device_map,
        torch_dtype=torch.bfloat16,
        max_memory=model_max_memory,
    )
    harness = EvaluationHarness(
        fur,
        LOG_PATH,
        min_lemmas_per_synset=MIN_LEMMAS,
        max_token_count=MAX_TOKENS,
    )

    used_synsets = sorted(
        {
            name
            for direction in active_directions
            for name in (direction["attract"], direction["repel"])
        }
    )
    for name in used_synsets:
        cached = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
        concepts[name] = Concept(
            synset=cached.synset, tensor=cached.tensor.to(fur.model.device)
        )
    print(f"loaded {len(concepts)} cached concepts")

    baseline_spec = {
        "model": MODEL_KEY,
        "steps": run_steps,
        "max_new_tokens": run_steps + 1,
        "prompts": {row["id"]: prompts_by_id[row["id"]] for row in active_prompts},
    }
    if BASELINE_PATH.exists():
        baseline_payload = json.loads(BASELINE_PATH.read_text())
        if baseline_payload.get("spec") != baseline_spec:
            raise RuntimeError(
                f"{BASELINE_PATH} does not match this run; use OVERWRITE=True"
            )
        baseline_texts_by_id = baseline_payload["texts"]
        print(f"reused {len(baseline_texts_by_id)} baselines")
    elif RUN_GENERATION:
        for prompt_row in active_prompts:
            prompt_id = prompt_row["id"]
            baseline_texts_by_id[prompt_id] = harness.generate_baseline(
                [prompts_by_id[prompt_id]], max_new_tokens=run_steps + 1
            )[0]
        BASELINE_PATH.write_text(
            json.dumps(
                {"spec": baseline_spec, "texts": baseline_texts_by_id},
                indent=2,
                ensure_ascii=False,
            )
            + "\n"
        )
        print(f"wrote {BASELINE_PATH}")
    else:
        raise FileNotFoundError(
            f"RUN_TRAJECTORIES requires an existing {BASELINE_PATH}"
        )
else:
    print("model load skipped")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

2026-08-10 20:43:10.525 | INFO     | frames.models.hf.base:__init__:89 - Loaded model: meta-llama/Llama-3.1-8B-Instruct


2026-08-10 20:43:10.527 | WARNING  | frames.models.hf.base:__init__:90 - memory cost: 15316 Mb


loaded 2 cached concepts


  0%|                                                     | 0/1 [00:00<?, ?it/s]

...

  0%|                                                     | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.06s/it]

100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.07s/it]

wrote resources/54_long_negative_baselines.json


## Guided run

All three guided conditions use the same lambda from YAML. `frame-diff`
applies the FRH Procrustes projection of attract minus repel. `score-neg`
retains separate guides and normalizes each concept over the candidate pool
before applying weights `[+1, -1]`.


In [6]:
FAMILY = {
    "attract-only": "positive-control",
    "frame-diff": "frame",
    "score-neg": "score",
}

def make_methods(prompt, direction, lam):
    attract = concepts[direction["attract"]]
    repel = concepts[direction["repel"]]
    differential = attract - repel
    return {
        "attract-only": lambda: fur.generate_with_topk_multi_guide(
            [prompt],
            guides=[attract],
            k=K,
            steps=run_steps,
            fluency_weight=lam,
            batch_size=1,
        ),
        "frame-diff": lambda: fur.generate_with_topk_multi_guide(
            [prompt],
            guides=[differential],
            k=K,
            steps=run_steps,
            fluency_weight=lam,
            batch_size=1,
        ),
        "score-neg": lambda: fur.generate_with_topk_multi_guide(
            [prompt],
            guides=[attract, repel],
            weights=[1.0, -1.0],
            k=K,
            steps=run_steps,
            normalize="zscore",
            fluency_weight=lam,
            batch_size=1,
        ),
    }


if RUN_GENERATION:
    for direction in active_directions:
        attract = concepts[direction["attract"]]
        repel = concepts[direction["repel"]]
        measured_rho = float(attract.rho(repel))
        for prompt_row in active_prompts:
            prompt_id = prompt_row["id"]
            prompt = prompts_by_id[prompt_id]
            for lam in active_lambdas:
                methods = make_methods(prompt, direction, lam)
                for method in METHODS:
                    key = (
                        direction["id"],
                        prompt_id,
                        method,
                        float(lam),
                        run_steps,
                    )
                    if key in completed:
                        continue
                    torch.cuda.empty_cache()
                    started = time.perf_counter()
                    texts, probe = methods[method]()
                    elapsed = time.perf_counter() - started
                    continuation = harness.continuation(texts[0], prompt)
                    metadata = generation_metadata(
                        fur.model.tokenizer, continuation, run_steps, elapsed
                    )
                    harness.evaluate(
                        [prompt],
                        texts,
                        [],  # presence is not an E54 endpoint; avoids cuDF here
                        config={
                            "experiment": "54",
                            "model": MODEL_KEY,
                            "direction_id": direction["id"],
                            "prompt_id": prompt_id,
                            "method": method,
                            "family": FAMILY[method],
                            "lam": float(lam),
                            "attract": direction["attract"],
                            "repel": direction["repel"],
                            "rho_model": measured_rho,
                            "k": K,
                            "steps": run_steps,
                            "survivor_rule": "greedy",
                        },
                        probe=probe,
                        baseline_texts=[baseline_texts_by_id[prompt_id]],
                        concepts={
                            direction["attract"]: attract,
                            direction["repel"]: repel,
                        },
                        angles=True,
                        metadata=[metadata],
                    )
                    completed.add(key)
                    print(
                        f"{len(completed):>2}/{active_cells} "
                        f"{direction['id']} / {prompt_id} / {method}: "
                        f"{metadata['token_count']} tokens, {elapsed:.1f}s"
                    )


  0%|                                                     | 0/1 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.57s/it]

100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.57s/it]

 1/60 woman_minus_man / men / attract-only: 27 tokens, 1.6s


  0%|                                                     | 0/1 [00:00<?, ?it/s]

...

59/60 man_minus_woman / happiness / frame-diff: 27 tokens, 1.5s


  0%|                                                     | 0/1 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.56s/it]

100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.56s/it]

60/60 man_minus_woman / happiness / score-neg: 27 tokens, 1.6s


## Whole-continuation results and controls

Clean success is deliberately strict: the attracted concept must rise and
the repelled concept must fall relative to the same unsteered baseline.
Synthetic `unsteered` rows (zero deltas, PPL ratio 1) are included in the
aggregate table; the raw JSONL remains one record per guided cell.


In [7]:
parsed = read_records(LOG_PATH)
result_rows = []
for record in parsed:
    config = record["config"]
    expression = record["expression"]
    angles = record["angles"]
    delta_attract = expression[config["attract"]]["delta"]
    delta_repel = expression[config["repel"]]["delta"]
    cos_attract = angles[config["attract"]]["delta"]["token_cos"]
    cos_repel = angles[config["repel"]]["delta"]["token_cos"]
    metadata = record.get("metadata", {})
    result_rows.append(
        {
            "direction_id": config["direction_id"],
            "prompt_id": config["prompt_id"],
            "method": config["method"],
            "lam": config["lam"],
            "attract": config["attract"],
            "repel": config["repel"],
            "delta_attract": delta_attract,
            "delta_repel": delta_repel,
            "net_expression": delta_attract - delta_repel,
            "clean_expression": delta_attract > 0 and delta_repel < 0,
            "cos_attract": cos_attract,
            "cos_repel": cos_repel,
            "net_token_cos": cos_attract - cos_repel,
            "clean_token_cos": cos_attract > 0 and cos_repel < 0,
            "ppl_ratio": record["ppl_ratio"],
            "continuation_tokens": metadata.get("token_count"),
            "termination": metadata.get("termination"),
            "distinct_token_ratio": metadata.get("distinct_token_ratio"),
            "repeated_4gram_rate": metadata.get("repeated_ngram_rate"),
            "max_repeated_span_tokens": metadata.get(
                "max_repeated_span_tokens"
            ),
            "elapsed_seconds": metadata.get("elapsed_seconds"),
        }
    )

results = pd.DataFrame(result_rows)
summary = pd.DataFrame()
analysis_results = results.copy()
if not results.empty:
    baseline_rows = []
    for direction in active_directions:
        for prompt_row in active_prompts:
            baseline_rows.append(
                {
                    "direction_id": direction["id"],
                    "prompt_id": prompt_row["id"],
                    "method": "unsteered",
                    "lam": active_lambdas[0],
                    "attract": direction["attract"],
                    "repel": direction["repel"],
                    "delta_attract": 0.0,
                    "delta_repel": 0.0,
                    "net_expression": 0.0,
                    "clean_expression": False,
                    "cos_attract": 0.0,
                    "cos_repel": 0.0,
                    "net_token_cos": 0.0,
                    "clean_token_cos": False,
                    "ppl_ratio": 1.0,
                }
            )
    analysis_results = pd.concat(
        [results, pd.DataFrame(baseline_rows)], ignore_index=True
    )
    summary = (
        analysis_results.groupby(
            ["direction_id", "attract", "repel", "method", "lam"],
            dropna=False,
        )
        .agg(
            delta_attract=("delta_attract", "mean"),
            delta_repel=("delta_repel", "mean"),
            net_expression=("net_expression", "mean"),
            clean_expression=("clean_expression", "mean"),
            cos_attract=("cos_attract", "mean"),
            cos_repel=("cos_repel", "mean"),
            net_token_cos=("net_token_cos", "mean"),
            clean_token_cos=("clean_token_cos", "mean"),
            ppl_ratio=("ppl_ratio", "mean"),
            n_prompts=("prompt_id", "nunique"),
        )
        .reset_index()
    )
    summary["observed_guided_records"] = len(results)
    summary["expected_guided_records"] = active_cells
    summary["complete"] = len(results) == active_cells
    summary.to_csv(SUMMARY_PATH, index=False)
    display(summary.round(4))
    print(f"wrote {SUMMARY_PATH}; completeness {len(results)}/{active_cells}")
else:
    print(f"no records at {LOG_PATH}; generation remains disabled")


,direction_id,attract,repel,method,lam,delta_attract,delta_repel,net_expression,clean_expression,cos_attract,cos_repel,net_token_cos,clean_token_cos,ppl_ratio,n_prompts,observed_guided_records,expected_guided_records,complete
0,man_minus_woman,man.n.01,woman.n.01,attract-only,0.4,1.1065,0.6826,0.4238,0.0,0.0110,0.0074,0.0036,0.1,2.2108,10,60,60,True
1,man_minus_woman,man.n.01,woman.n.01,frame-diff,0.4,0.0245,-0.5188,0.5433,0.3,0.0001,-0.0036,0.0037,0.1,2.0640,10,60,60,True
2,man_minus_woman,man.n.01,woman.n.01,score-neg,0.4,-0.3691,-0.8667,0.4976,0.4,0.0021,-0.0003,0.0024,0.0,3.4912,10,60,60,True
3,man_minus_woman,man.n.01,woman.n.01,unsteered,0.4,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,0.0000,0.0,1.0000,10,60,60,True
4,woman_minus_man,woman.n.01,man.n.01,attract-only,0.4,0.3116,0.1881,0.1236,0.1,0.0040,0.0041,-0.0001,0.0,2.3090,10,60,60,True
5,woman_minus_man,woman.n.01,man.n.01,frame-diff,0.4,0.0813,-0.2824,0.3636,0.2,0.0057,0.0044,0.0012,0.1,1.8755,10,60,60,True
6,woman_minus_man,woman.n.01,man.n.01,score-neg,0.4,-0.6486,-1.0454,0.3969,0.1,0.0051,0.0030,0.0022,0.1,17.8604,10,60,60,True
7,woman_minus_man,woman.n.01,man.n.01,unsteered,0.4,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,0.0000,0.0,1.0000,10,60,60,True


wrote resources/54_long_negative_summary.csv; completeness 60/60


In [8]:
if not results.empty:
    paired = results.pivot_table(
        index=["direction_id", "prompt_id"],
        columns="method",
        values="delta_repel",
        aggfunc="first",
    )
    suppression_rows = []
    for method in ("frame-diff", "score-neg"):
        comparison = paired[method] - paired["attract-only"]
        for direction_id, values in comparison.groupby(level="direction_id"):
            suppression_rows.append(
                {
                    "direction_id": direction_id,
                    "method": method,
                    "prompts_more_suppressed": int((values < 0).sum()),
                    "n_prompts": int(values.notna().sum()),
                    "mean_delta_repel_vs_attract_only": values.mean(),
                }
            )
    suppression = pd.DataFrame(suppression_rows)
    print("Paired suppression relative to the positive-only control:")
    display(suppression.round(4))

    symmetry = summary[summary["method"] != "unsteered"].pivot_table(
        index="method",
        columns="direction_id",
        values=["net_expression", "clean_expression", "net_token_cos"],
    )
    print("Direction asymmetry (directions remain separate):")
    display(symmetry.round(4))


Paired suppression relative to the positive-only control:


,direction_id,method,prompts_more_suppressed,n_prompts,mean_delta_repel_vs_attract_only
0,man_minus_woman,frame-diff,9,10,-1.2014
1,woman_minus_man,frame-diff,7,10,-0.4704
2,man_minus_woman,score-neg,10,10,-1.5493
3,woman_minus_man,score-neg,9,10,-1.2335


Direction asymmetry (directions remain separate):


clean_expression                  net_expression                  \
direction_id  man_minus_woman woman_minus_man man_minus_woman woman_minus_man   
method                                                                          
attract-only              0.0             0.1          0.4238          0.1236   
frame-diff                0.3             0.2          0.5433          0.3636   
score-neg                 0.4             0.1          0.4976          0.3969   

               net_token_cos                  
direction_id man_minus_woman woman_minus_man  
method                                        
attract-only          0.0036         -0.0001  
frame-diff            0.0037          0.0012  
score-neg             0.0024          0.0022

## Optional 20-token trajectories

At 24 steps this normally yields one full 20-token bin and one short final
bin. Absolute steered/baseline values, coverage, attraction, repulsion, net
contrast, clean success, and segment PPL are retained. No missing tokens are
padded into either continuation.


In [9]:
def token_hidden(text):
    tokens = fur.model.tokenize(text)
    with torch.inference_mode():
        hidden = fur.model.forward_last_hiden_state(tokens).squeeze(0)
    ids = tokens.squeeze(0).tolist()
    bos_id = fur.model.tokenizer.bos_token_id
    if ids and ids[0] == bos_id:
        ids = ids[1:]
        hidden = hidden[1:]
    return ids, hidden


def binned_values(ids, hidden, concept_map):
    bins = token_bins(ids, BIN_WIDTH)
    if not bins:
        return {}
    values = {part.index: {"bin": part} for part in bins}
    for name, concept in concept_map.items():
        projected = fur._project_hidden_states(
            hidden.unsqueeze(0), concept
        ).reshape(len(ids), -1).float().mean(-1)
        cosine = vector_subspace_cosine(
            hidden, concept.tensor.squeeze(0)
        ).float()
        for part in bins:
            values[part.index][name] = {
                "expression": float(projected[part.start : part.stop].mean()),
                "token_cos": float(cosine[part.start : part.stop].mean()),
            }
    return values


def segment_perplexity(token_ids):
    if len(token_ids) < 2:
        return float("nan")
    return harness.perplexity(fur.model.tokenizer.decode(token_ids))


trajectory_rows = []
if RUN_TRAJECTORIES:
    baseline_hidden = {}
    baseline_ppl = {}
    for record_index, record in enumerate(parsed, start=1):
        config = record["config"]
        prompt_id = config["prompt_id"]
        concept_map = {
            config["attract"]: concepts[config["attract"]],
            config["repel"]: concepts[config["repel"]],
        }
        steered_ids, steered_hidden = token_hidden(record["continuation"])
        steered = binned_values(steered_ids, steered_hidden, concept_map)
        if prompt_id not in baseline_hidden:
            baseline_continuation = harness.continuation(
                baseline_texts_by_id[prompt_id], prompts_by_id[prompt_id]
            )
            baseline_hidden[prompt_id] = token_hidden(baseline_continuation)
        base_ids, base_hidden = baseline_hidden[prompt_id]
        baseline = binned_values(base_ids, base_hidden, concept_map)

        for bin_index in sorted(set(steered) & set(baseline)):
            sbin = steered[bin_index]["bin"]
            bbin = baseline[bin_index]["bin"]
            bppl_key = (prompt_id, bin_index)
            if bppl_key not in baseline_ppl:
                baseline_ppl[bppl_key] = segment_perplexity(bbin.token_ids)
            segment_ppl = segment_perplexity(sbin.token_ids)
            attract = config["attract"]
            repel = config["repel"]
            attr_expr = (
                steered[bin_index][attract]["expression"]
                - baseline[bin_index][attract]["expression"]
            )
            repel_expr = (
                steered[bin_index][repel]["expression"]
                - baseline[bin_index][repel]["expression"]
            )
            attr_cos = (
                steered[bin_index][attract]["token_cos"]
                - baseline[bin_index][attract]["token_cos"]
            )
            repel_cos = (
                steered[bin_index][repel]["token_cos"]
                - baseline[bin_index][repel]["token_cos"]
            )
            trajectory_rows.append(
                {
                    "direction_id": config["direction_id"],
                    "prompt_id": prompt_id,
                    "method": config["method"],
                    "lam": config["lam"],
                    "bin": bin_index,
                    "start_token": sbin.start,
                    "stop_token": sbin.stop,
                    "coverage": min(len(sbin.token_ids), len(bbin.token_ids))
                    / BIN_WIDTH,
                    "attract_expression_steered": steered[bin_index][attract][
                        "expression"
                    ],
                    "attract_expression_baseline": baseline[bin_index][attract][
                        "expression"
                    ],
                    "attract_expression_delta": attr_expr,
                    "repel_expression_steered": steered[bin_index][repel][
                        "expression"
                    ],
                    "repel_expression_baseline": baseline[bin_index][repel][
                        "expression"
                    ],
                    "repel_expression_delta": repel_expr,
                    "net_expression": attr_expr - repel_expr,
                    "clean_expression": attr_expr > 0 and repel_expr < 0,
                    "attract_token_cos_delta": attr_cos,
                    "repel_token_cos_delta": repel_cos,
                    "net_token_cos": attr_cos - repel_cos,
                    "clean_token_cos": attr_cos > 0 and repel_cos < 0,
                    "segment_ppl": segment_ppl,
                    "baseline_segment_ppl": baseline_ppl[bppl_key],
                    "segment_ppl_ratio": segment_ppl / baseline_ppl[bppl_key],
                }
            )
        del steered_hidden
        if record_index % 10 == 0:
            torch.cuda.empty_cache()
            print(f"trajectory {record_index}/{len(parsed)}")

    trajectories = pd.DataFrame(trajectory_rows)
    trajectories.to_csv(TRAJECTORY_PATH, index=False)
    print(f"wrote {len(trajectories)} rows to {TRAJECTORY_PATH}")
    if not trajectories.empty:
        trajectory_summary = (
            trajectories.groupby(["direction_id", "method", "bin"])
            .agg(
                net_expression=("net_expression", "mean"),
                clean_expression=("clean_expression", "mean"),
                net_token_cos=("net_token_cos", "mean"),
                segment_ppl_ratio=("segment_ppl_ratio", "mean"),
                coverage=("coverage", "mean"),
            )
            .reset_index()
        )
        fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2))
        for axis, metric, ylabel in zip(
            axes,
            ("net_expression", "clean_expression", "segment_ppl_ratio"),
            ("net expression", "clean-success rate", "segment PPL ratio"),
        ):
            for (direction_id, method), cell in trajectory_summary.groupby(
                ["direction_id", "method"]
            ):
                axis.plot(
                    cell["bin"] * BIN_WIDTH,
                    cell[metric],
                    marker="o",
                    label=f"{direction_id}: {method}",
                )
            axis.set_xlabel("continuation token position")
            axis.set_ylabel(ylabel)
        axes[0].axhline(0, color="gray", linewidth=0.8)
        axes[0].legend(fontsize=7)
        plt.tight_layout()
        plt.savefig(TRAJECTORY_FIGURE_PATH, dpi=150)
        plt.show()

        final_bins = (
            trajectories.sort_values("bin")
            .groupby(["direction_id", "prompt_id", "method", "lam"])
            .tail(1)
            .groupby(["direction_id", "method", "lam"])
            .agg(
                final_net_expression=("net_expression", "mean"),
                final_clean_expression=("clean_expression", "mean"),
                final_net_token_cos=("net_token_cos", "mean"),
                final_segment_ppl_ratio=("segment_ppl_ratio", "mean"),
                final_bin_coverage=("coverage", "mean"),
            )
            .reset_index()
        )
        summary = summary.merge(
            final_bins, on=["direction_id", "method", "lam"], how="left"
        )
        summary.to_csv(SUMMARY_PATH, index=False)


trajectory 10/60


trajectory 20/60


trajectory 30/60


trajectory 40/60


trajectory 50/60


trajectory 60/60
wrote 112 rows to resources/54_long_negative_trajectories.csv


In [10]:
if not results.empty:
    plot_rows = summary[summary["method"] != "unsteered"]
    directions = list(plot_rows["direction_id"].unique())
    fig, axes = plt.subplots(1, len(directions), figsize=(6 * len(directions), 4.8))
    if len(directions) == 1:
        axes = [axes]
    colors = {
        "attract-only": "tab:gray",
        "frame-diff": "tab:blue",
        "score-neg": "tab:orange",
    }
    for axis, direction_id in zip(axes, directions):
        cell = plot_rows[plot_rows["direction_id"] == direction_id]
        for _, row in cell.iterrows():
            axis.scatter(
                row["delta_repel"],
                row["delta_attract"],
                s=90,
                color=colors[row["method"]],
                label=row["method"],
            )
            axis.annotate(
                row["method"],
                (row["delta_repel"], row["delta_attract"]),
                xytext=(5, 3),
                textcoords="offset points",
                fontsize=8,
            )
        axis.axhline(0, color="gray", linewidth=0.8)
        axis.axvline(0, color="gray", linewidth=0.8)
        axis.set_title(direction_id)
        axis.set_xlabel("repelled-concept expression delta (lower is better)")
    axes[0].set_ylabel("attracted-concept expression delta (higher is better)")
    plt.tight_layout()
    plt.savefig(FIGURE_PATH, dpi=150)
    plt.show()

    qualitative = []
    for record in parsed:
        config = record["config"]
        qualitative.append(
            {
                "direction": config["direction_id"],
                "prompt": config["prompt_id"],
                "method": config["method"],
                "continuation": record["continuation"],
                "stereotype_review": "",
            }
        )
    print(
        "Review outputs for stereotypes and asymmetric social claims; "
        "leave the review column descriptive."
    )
    display(pd.DataFrame(qualitative))


Review outputs for stereotypes and asymmetric social claims; leave the review column descriptive.


,direction,prompt,method,continuation,stereotype_review
0,woman_minus_man,men,attract-only,1. Husbands. 1. Brothers. 1. Sons. 1. Fathers....,
1,woman_minus_man,men,frame-diff,1. Men can have a strong physical presence.\n2...,
2,woman_minus_man,men,score-neg,1. A father.,
3,woman_minus_man,women,attract-only,1. A mother.2. Sister3. Wife4. Daughter5. Gran...,
4,woman_minus_man,women,frame-diff,"1. A mother, a daughter, a sister, a friend, a...",
5,woman_minus_man,women,score-neg,"1. A mother, daughter, sister, aunt, grandmoth...",
6,woman_minus_man,short_story,attract-only,"Once, a young artist named Lily spent her enti...",
7,woman_minus_man,short_story,frame-diff,"Once, there was a young girl named Akira who l...",
8,woman_minus_man,short_story,score-neg,"Once, there was a tiny island, surrounded by a...",
9,woman_minus_man,favorite_place,attract-only,"My favorite place is a digital construct, I do...",
